<div style="display:flex; justify-content:space-between; align-items:center; width:100%; margin:8px 0 24px 0;"><div style="text-align:left;"><img src="https://www.ec-nantes.fr/medias/photo/logocn-rvb_1648479844750-png?ID_FICHE=178994&amp;INLINE=FALSE" alt="Centrale Nantes" style="height:72px; width:auto;"></div><div style="text-align:right; font-size:18px; font-weight:600; color:#17324d; line-height:1.35;">MSc. CORO DASSIP</div></div>
<div style="border:2px solid #333; padding:14px 20px; margin:15px auto 25px auto; width:85%; max-width:900px; box-sizing:border-box; text-align:center;"><h1 style="margin:0;"><b>Image Segmentation — Algorithm</b></h1></div>


## Purpose and Reading Convention

This notebook explains how to implement and review this lab inside its existing `labs/` directory. The executable code remains in [main.ipynb](main.ipynb); no repository split or Python package extraction is required.

Each numbered step gives **Purpose**, **Inputs**, **Procedure**, **Outputs**, **Validation** and **On failure**. Follow dependencies in order and reuse the same intermediate arrays for metrics and figures.

**Validation convention:** these are review contracts. Some are enforced by existing assertions or exceptions; others require inspecting intermediate results. A documentation update does not add runtime guards. The *Implementation limits* section identifies important gaps so the algorithm does not claim unimplemented protections.

Use parameter values and numeric tolerances actually declared in `main.ipynb`. Do not replace them with unsupported universal thresholds. A failed implementation check stops the run; a diagnostic limitation must be reported rather than silently treated as a pass.

## Inputs and Final Outputs

| Item | Description |
| --- | --- |
| **Inputs** | Hand, tower, and peppers images |
| **Final outputs** | Binary/region segmentations, region measurements, evaluation metrics, method-selection diagnostics, 21 required figures |
| **Implementation target** | `main.ipynb` |

## 0. Start

**Purpose**

Establish the scope and a clean execution order.

**Inputs**

The problem statement, theory, requirements and main notebook in this lab.

**Procedure**

1. Select the project kernel and use this notebook as the coding plan for `main.ipynb`.
2. Adopt one consistent foreground-mask convention across all methods.
3. Keep controlled metric demonstrations distinct from external ground truth.

**Outputs**

Selected environment and a top-to-bottom execution plan.

**Validation**

Review the local assumptions before using cached notebook outputs.

**On failure**

Resolve missing prerequisites first; do not infer a successful run from stored outputs.

## 1. Create Setup, Paths, and Image Roles

**Purpose**

Assign fixed roles to input images.

**Inputs**

Hand, tower and peppers files.

**Procedure**

1. Import `Path`, NumPy, Matplotlib, PIL, SciPy `ndimage`, and OpenCV.
2. Locate the lab root and create data/output paths.
3. Discover supported images and map hand, tower, and peppers roles deterministically.
4. Create reusable grayscale/RGB loading and figure-saving helpers.

**Outputs**

Paths, loaders and figure helper.

**Validation**

Each role resolves; grayscale/RGB conversions are explicit.

**On failure**

Stop if a role is missing rather than replacing the scene.

## 2. Define Mask Conversion and Overlay Helpers

**Purpose**

Separate logical masks from image encodings.

**Inputs**

Boolean mask and same-sized RGB image.

**Procedure**

1. Keep logical masks as boolean arrays inside the notebook.
2. Convert to uint8 0/255 only at OpenCV boundaries.
3. Create an overlay helper that highlights foreground on RGB images.
4. Verify mask and image spatial dimensions match.

**Outputs**

0/255 conversion and overlay functions.

**Validation**

Mask spatial shape matches image; conversion preserves foreground.

**On failure**

Correct shape/channel mismatches before overlaying.

## 3. Define the Thresholding Convention

**Purpose**

Keep threshold polarity consistent.

**Inputs**

2-D grayscale image, threshold and polarity.

**Procedure**

1. Create a helper accepting grayscale image, threshold, and foreground polarity.
2. For dark foreground return `image<threshold`; for bright foreground return `image>=threshold`.
3. Reject unsupported polarity modes.
4. Reuse this helper for manual-threshold baselines.

**Outputs**

Boolean threshold mask.

**Validation**

Dark means value<threshold; bright means value>=threshold.

**On failure**

Existing helper rejects unsupported polarity and non-2-D input.

## 4. Load the Three Lab Images

**Purpose**

Confirm the experiment inputs.

**Inputs**

Resolved image paths.

**Procedure**

1. Load the hand as grayscale and tower/peppers as RGB.
2. Verify dimensions, dtypes, and non-empty arrays.
3. Display or save one common source-image overview.

**Outputs**

Hand grayscale; tower/peppers RGB; source overview.

**Validation**

uint8 nonempty arrays with appropriate dimensions.

**On failure**

Stop on decoding errors or wrong image roles.

## 5. Inspect the Hand Histogram

**Purpose**

Support the manual threshold with intensity evidence.

**Inputs**

8-bit hand image.

**Procedure**

1. Flatten the hand image and compute the full 8-bit histogram.
2. Use the same intensity range as later global thresholds.
3. Display the histogram as evidence for the initial manual threshold.

**Outputs**

256-bin histogram.

**Validation**

Total count equals image pixels; same intensity range as thresholding.

**On failure**

Correct histogram range before choosing the threshold.

## 6. Create a Manual Global-Threshold Baseline

**Purpose**

Establish a simple interpretable baseline.

**Inputs**

Hand image and declared threshold.

**Procedure**

1. Choose the declared manual threshold.
2. Create a dark-foreground boolean mask.
3. Display original, mask, and overlay.
4. Save this baseline before automatic methods.

**Outputs**

Dark-foreground mask and overlay.

**Validation**

Boolean mask preserves spatial shape; inspect hand/background polarity.

**On failure**

Correct inversion before comparing automatic methods.

## 7. Test Threshold Sensitivity

**Purpose**

Expose instability around the manual choice.

**Inputs**

Same image and nearby thresholds.

**Procedure**

1. Choose several thresholds around the baseline.
2. Apply identical foreground semantics to all.
3. Display masks with identical layout.
4. Use the comparison to reveal unstable regions.

**Outputs**

Threshold-sensitivity grid.

**Validation**

Only threshold varies; foreground convention unchanged.

**On failure**

Do not interpret polarity changes as threshold sensitivity.

## 8. Apply Otsu Thresholding

**Purpose**

Choose a global threshold from the histogram.

**Inputs**

Hand grayscale array.

**Procedure**

1. Call OpenCV thresholding with `THRESH_BINARY_INV + THRESH_OTSU` because the hand is darker foreground.
2. Record the returned Otsu threshold.
3. Convert the OpenCV mask to boolean.
4. Display and save the result.

**Outputs**

Otsu threshold and boolean mask.

**Validation**

Use inverted binary output for dark hand foreground.

**On failure**

Inspect a poor mask as a limitation of global separation, not a failed function call.

## 9. Smooth Before Otsu

**Purpose**

Test noise reduction before threshold estimation.

**Inputs**

Hand image and fixed Gaussian kernel.

**Procedure**

1. Apply a small Gaussian blur to the hand image.
2. Recompute Otsu on the smoothed image.
3. Compare threshold and mask against unsmoothed Otsu.
4. Keep blur kernel fixed and explicit.

**Outputs**

Smoothed image, new Otsu threshold and mask.

**Validation**

Compare against unsmoothed Otsu on the same source.

**On failure**

Report erased thin details; blur is not automatically beneficial.

## 10. Compare Adaptive Thresholds

**Purpose**

Compare local threshold estimators.

**Inputs**

Same source, odd block size>1 and offset.

**Procedure**

1. Apply adaptive mean and adaptive Gaussian thresholding.
2. Use the same block size and offset for a fair comparison.
3. Use inverted binary output to preserve foreground semantics.
4. Compare both masks with global methods.

**Outputs**

Adaptive mean/Gaussian masks.

**Validation**

Same block/offset and inverted polarity for both.

**On failure**

Correct invalid block settings; inspect texture-induced foreground.

## 11. Implement a Morphology Helper

**Purpose**

Reuse morphological operations with one encoding.

**Inputs**

Mask, operation, kernel and iterations.

**Procedure**

1. Accept mask, operation name, structuring element, and iteration count.
2. Convert boolean mask to uint8 once at the OpenCV boundary.
3. Route `erode`, `dilate`, `open`, and `close` correctly.
4. Return a boolean mask and reject invalid operations or iteration counts.

**Outputs**

Boolean result mask.

**Validation**

Supported operations erode/dilate/open/close; iterations>=1.

**On failure**

Existing helper raises for unknown operation or invalid iterations.

## 12. Create Structuring Elements

**Purpose**

Make neighborhood shape explicit.

**Inputs**

Common nominal kernel size.

**Procedure**

1. Build rectangular, elliptical, and cross kernels with the same nominal size.
2. Visualize their supports.
3. Use this to make shape assumptions explicit before morphology.

**Outputs**

Rectangular, ellipse and cross supports.

**Validation**

Supports are nonempty and share declared dimensions.

**On failure**

Correct kernel shape before interpreting topology changes.

## 13. Apply Erosion and Dilation

**Purpose**

Measure foreground shrinkage and growth.

**Inputs**

Working mask and same structuring element.

**Procedure**

1. Convert the working mask to uint8.
2. Apply one erosion and one dilation with the chosen kernel.
3. Convert outputs back to boolean.
4. Compare how foreground extent changes.

**Outputs**

Eroded and dilated masks.

**Validation**

Same source, kernel and iteration count; foreground convention fixed.

**On failure**

Check inverted polarity if the interpretation appears reversed.

## 14. Apply Opening and Closing

**Purpose**

Compare removal of small foreground with closure of gaps.

**Inputs**

Same starting mask and kernel.

**Procedure**

1. Use opening to remove small foreground structures.
2. Use closing to fill small gaps/connect nearby foreground structures.
3. Apply both to the same starting mask with the same kernel.
4. Save the comparison.

**Outputs**

Opening/closing comparison.

**Validation**

Do not feed one comparison branch into the other.

**On failure**

Reduce kernel size if valid structures disappear or merge excessively.

## 15. Compute the Morphological Gradient

**Purpose**

Expose the morphological boundary band.

**Inputs**

Binary mask and chosen kernel.

**Procedure**

1. Apply OpenCV morphological gradient to the binary mask.
2. Interpret the result as a boundary-focused representation.
3. Display mask and gradient together.

**Outputs**

Morphological gradient.

**Validation**

Compute dilation-minus-erosion through OpenCV; preserve dimensions.

**On failure**

Treat this as a boundary representation, not a region label map.

## 16. Fill Holes

**Purpose**

Fill enclosed background regions.

**Inputs**

Working boolean mask.

**Procedure**

1. Convert the working mask to boolean.
2. Apply SciPy `binary_fill_holes`.
3. Compare before/after.
4. Pass only the filled boolean mask to connected-component analysis.

**Outputs**

Hole-filled boolean mask.

**Validation**

Same shape; existing foreground retained.

**On failure**

Inspect topology: filling real holes may be inappropriate for another object.

## 17. Label Connected Components

**Purpose**

Assign identities to connected foreground regions.

**Inputs**

Filled mask encoded as uint8.

**Procedure**

1. Convert the filled mask to uint8.
2. Call `connectedComponentsWithStats` with 8-connectivity.
3. Treat label 0 as background.
4. Extract foreground component areas from the returned statistics.

**Outputs**

Labels, statistics, centroids and component count.

**Validation**

8-connectivity; label 0 is background; stats align with label IDs.

**On failure**

Do not include background area among object measurements.

## 18. Remove Small Components

**Purpose**

Remove regions by a stated pixel-area rule.

**Inputs**

Component labels/stats and minimum area.

**Procedure**

1. Create an empty boolean result mask.
2. For each foreground label read its area.
3. Keep only labels whose area meets the minimum threshold.
4. Combine retained labels into the cleaned mask.

**Outputs**

Cleaned boolean mask.

**Validation**

Keep foreground labels with area>=threshold; never keep label 0.

**On failure**

If all foreground disappears, report the parameter effect rather than fabricate an object.

## 19. Extract External Contours

**Purpose**

Trace external object boundaries.

**Inputs**

Cleaned mask.

**Procedure**

1. Convert the cleaned mask to uint8.
2. Call `findContours` with external retrieval.
3. Draw all contours on an RGB copy of the hand image.
4. Save the contour overlay.

**Outputs**

External contours and RGB overlay.

**Validation**

Mask converted to uint8; retrieval excludes internal holes intentionally.

**On failure**

Handle an empty contour list; it means no detected region.

## 20. Measure Region Properties

**Purpose**

Quantify the extracted regions.

**Inputs**

Contours.

**Procedure**

1. For each contour compute area, perimeter, and bounding rectangle.
2. Compute any declared compactness/shape quantity with zero-division protection.
3. Store one structured record per region.
4. Print the region table consistently.

**Outputs**

Area, perimeter, bounding box and shape records.

**Validation**

Guard zero perimeter; contour area is not identical to pixel-count area.

**On failure**

Skip or mark undefined shape measures for degenerate contours.

## 21. Segment by Color in HSV

**Purpose**

Use hue separation with wraparound for red.

**Inputs**

RGB peppers converted to OpenCV HSV.

**Procedure**

1. Convert peppers from RGB to HSV.
2. Inspect hue, saturation, and value channels.
3. Define two red hue intervals because red wraps around the hue axis.
4. Threshold both ranges with minimum saturation/value constraints and combine masks.
5. Clean or overlay the final color mask as required.

**Outputs**

Two red-range masks and their union.

**Validation**

OpenCV 8-bit hue uses 0..179; saturation/value suppress achromatic/dark pixels.

**On failure**

Correct RGB/BGR and hue units before tuning ranges.

## 22. Create an Edge-Based Segmentation Example

**Purpose**

Build a boundary-based diagnostic.

**Inputs**

Tower grayscale, blur and Canny thresholds.

**Procedure**

1. Convert tower to grayscale.
2. Apply Gaussian smoothing.
3. Run Canny with explicit lower and upper thresholds.
4. Use morphology if needed to connect fragmented edges.
5. Save the edge-based diagnostic.

**Outputs**

Edges and morphological connection example.

**Validation**

Lower/upper thresholds ordered; source dimensions preserved.

**On failure**

Do not treat an edge map alone as filled object segmentation.

## 23. Compute the Distance Transform

**Purpose**

Measure foreground distance to background.

**Inputs**

Cleaned binary foreground mask.

**Procedure**

1. Convert the cleaned foreground mask to uint8.
2. Compute Euclidean distance to background.
3. Display distance map beside the binary mask.
4. Use high-distance interior pixels as marker evidence.

**Outputs**

Nonnegative distance map.

**Validation**

Background distance is zero; larger values indicate interior points.

**On failure**

Inspect empty/all-foreground masks before using marker thresholds.

## 24. Run Watershed Segmentation

**Purpose**

Separate regions using labelled markers.

**Inputs**

Peppers image, initial mask and distance map.

**Procedure**

1. Choose peppers as the watershed source.
2. Create an initial binary foreground estimate.
3. Use morphology to obtain sure background.
4. Threshold the distance transform for sure foreground.
5. Label foreground markers and mark unknown regions.
6. Run OpenCV watershed on a compatible color image.
7. Treat label `-1` as watershed boundary and visualize the separated regions.

**Outputs**

Watershed labels and boundary overlay.

**Validation**

Markers int32; unknown=0 before watershed; boundary=-1 afterward; image is 3-channel.

**On failure**

Inspect missing/merged markers if objects remain joined or fragment.

## 25. Implement Segmentation Metrics

**Purpose**

Define overlap metrics through explicit confusion counts.

**Inputs**

Same-shaped reference and predicted masks.

**Procedure**

1. Require reference and prediction to have the same spatial shape; review this precondition because the current helper does not explicitly check it.
2. Convert both masks to boolean and count TP, TN, FP and FN with logical operations.
3. Compute accuracy, precision, recall, IoU and Dice using the current `1e-12` denominator epsilon.
4. Explain undefined/empty-mask conventions explicitly: the implementation returns zero for zero-overlap denominators instead of assigning perfect similarity to an empty pair.

**Outputs**

TP,TN,FP,FN and five metrics.

**Validation**

Boolean masks; counts partition all pixels; epsilon=1e-12 in denominators.

**On failure**

Require equal shapes to avoid broadcasting; document empty-mask conventions rather than claim perfect overlap automatically.

## 26. Create a Controlled Metric Demonstration

**Purpose**

Demonstrate metric behavior with a controlled reference.

**Inputs**

Cleaned mask and morphological perturbation.

**Procedure**

1. Use the cleaned mask as a controlled reference.
2. Create a perturbed prediction by erosion or dilation.
3. Compute all metrics.
4. State clearly this is a metric demonstration, not external annotated ground truth.

**Outputs**

Metric example.

**Validation**

Reference is derived from this pipeline, not independent annotation.

**On failure**

Do not report these scores as real-world segmentation accuracy.

## 27. Verify the Dice–IoU Relationship

**Purpose**

Check the algebraic relation between overlap measures.

**Inputs**

IoU and Dice from the same pair.

**Procedure**

1. Read IoU and Dice from the same mask pair.
2. Compute Dice from IoU and IoU from Dice analytically.
3. Assert reconstructed values match direct measurements.

**Outputs**

Reconstructed Dice/IoU values.

**Validation**

Dice=2*IoU/(1+IoU); IoU=Dice/(2-Dice), compared with isclose.

**On failure**

Check denominator conventions and mask pairing on mismatch.

## 28. Compare Under- and Over-Segmentation

**Purpose**

Separate missed foreground from excess foreground.

**Inputs**

Same nonempty reference and erosion/dilation perturbations.

**Procedure**

1. Create an eroded version of the reference mask to simulate missing foreground.
2. Create a dilated version to simulate excess foreground.
3. Compute the same metrics for both.
4. Compare precision/recall behaviour using the unchanged reference mask.

**Outputs**

Under/over-segmentation metrics.

**Validation**

Compare precision and recall while keeping reference fixed.

**On failure**

Inspect empty masks before generalizing the expected trend.

## 29. Implement the End-to-End Dark-Foreground Pipeline

**Purpose**

Combine dark-object operations in a reusable order.

**Inputs**

2-D grayscale image, blur kernel and minimum area.

**Procedure**

1. Create a function receiving grayscale image, blur kernel, and minimum component area.
2. Gaussian-blur the image.
3. Estimate dark foreground using inverted Otsu thresholding.
4. Apply morphology, fill holes, label components, and remove small components.
5. Return threshold, intermediate masks, and final boolean mask in a structured result.

**Outputs**

Dictionary: threshold, blurred, binary, cleaned, filled, mask.

**Validation**

Otsu inverse, opening/closing, hole filling and area filtering preserve spatial shape.

**On failure**

Inspect intermediate masks to locate failure; do not tune only the final overlay.

## 30. Implement Method-Selection Logic

**Purpose**

Explain method choice from explicit scene properties.

**Inputs**

Flags for touching objects, color and illumination.

**Procedure**

1. Create a decision helper based on uneven illumination, discriminative color, and touching objects.
2. If objects touch, select distance transform plus marker-controlled watershed.
3. Else if color is discriminative, select color-space thresholding.
4. Else if illumination is uneven, select adaptive thresholding.
5. Otherwise select global/Otsu thresholding.

**Outputs**

Recommended strategy string.

**Validation**

Priority: touching objects, then color, then uneven illumination, then global threshold.

**On failure**

This helper recommends a strategy; it does not run the chosen algorithm.

## 31. Run the Integrated Segmentation Workflow

**Purpose**

Present the complete hand experiment.

**Inputs**

Hand image and dark-foreground pipeline.

**Procedure**

1. Run the reusable dark-foreground pipeline on the hand image.
2. Build the final colored overlay.
3. Display major intermediate stages and final result in one figure.
4. Save the integrated workflow output.

**Outputs**

Intermediate grid and final overlay.

**Validation**

Final overlay uses the returned mask and the same source.

**On failure**

Diagnose empty/overfilled regions through stored intermediate masks.

## 32. Run Final Validation

**Purpose**

Check masks, metric identities and figures.

**Inputs**

Computed masks, metrics and REQUIRED_OUTPUTS.

**Procedure**

1. Verify masks preserve source shape and boolean semantics where expected.
2. Verify metric values are finite and in valid ranges.
3. Verify Dice/IoU consistency checks pass.
4. Verify all declared output files exist.
5. Print the final pass message only when all checks succeed.

**Outputs**

Final validation outcome.

**Validation**

Same-shaped boolean masks; bounded finite metrics; Dice/IoU checks; 21 required PNGs.

**On failure**

Stop on actual failed assertions/missing files; passing them does not establish accuracy without external ground truth.

## Complete Formal Pseudocode

**Input:** Hand grayscale, tower RGB and peppers RGB; threshold/morphology/area/marker parameters.

**Output:** Boolean masks, connected components, contours, watershed labels, metric demonstrations, workflow result and 21 figures.

**Data contract:** Logical foreground is TRUE. OpenCV binary masks use uint8 {0,255}. Watershed marker labels are int32, with zero unknown before execution and −1 boundary afterward. 8-bit OpenCV hue spans 0…179.

The following numbered blocks form **one complete execution algorithm**. They include every lab-specific helper body, every numerical expression, every loop/branch and the existing validation/diagnostic steps. Inputs, literal parameters and output paths are assigned explicitly in the setup blocks; intermediate variables are produced before use in execution order. Each procedure's `RETURN` identifies its result. Top-level results are named by the output contract above.

**Scope:** this is a formal transcription of the current executable workflow, including its explicitly documented limitations. Additional physical-validity requirements in the surrounding review text are not silently presented as implemented guards. Error branches below specify the actual termination/rejection behaviour; the implementation-limit section identifies missing protections.

### Pseudocode notation and execution rules

- `x ← expression` assigns a value; `=` compares values. `A · B` is matrix multiplication; `*`, `/` and `^` are elementwise arithmetic for arrays and scalar arithmetic for numbers. Explicit dtype conversions are retained because clipping and overflow are part of these labs.
- Arrays are **zero-indexed**. `a:b:s` is an end-exclusive slice with step `s`; omitted limits use the array boundary. `RANGE(n)` produces `0,…,n−1`; `RANGE(a,b,s)` excludes `b`. Tuple assignment unpacks corresponding values. `TRANSPOSE(A)` interchanges axes as the array transpose operation does.
- `LIST[expression FOR x IN sequence WHERE condition]` constructs a list in iteration order. `STREAM[...]` is lazy; `SET{...}` deduplicates; `MAP{key ↦ value FOR ...}` creates a mapping. `EXPAND` inserts a sequence or mapping's members. `FUNCTION(parameters) ↦ expression` defines a callable expression.
- `IF`, `FOR`, `WHILE`, `RETURN`, `BREAK` and `CONTINUE` have their usual structured meanings and explicit closing statements. `REQUIRE condition OTHERWISE ...` stops on failure. `RAISE Error(message)` propagates that error; it is not a successful return. A logged frame/view rejection followed by `CONTINUE` is deliberately different from stopping the experiment.
- `AND`/`OR` short-circuit scalar conditions. `AND_EACH`/`OR_EACH` are elementwise operations for boolean masks and bitwise operations for integer flags. `COMPLEMENT` complements boolean masks or integer bits according to dtype. `NULL`, `TRUE`, `FALSE`, `ARRAY.nan` and `ARRAY.inf` retain their distinct meanings; `IS` tests identity.
- `PROCEDURE` defines the complete body shown below; indentation determines scope. Default arguments refer to the environment at definition time. `CLASS` defines stateful objects, `self` is the current instance, `__init__` initializes it, and `STATIC` methods need no instance. `NO_GRADIENT` disables automatic differentiation throughout the procedure. Each block continues the shared state of the preceding block; run them **once, top to bottom**, starting with a fresh environment.
- `CAST`, `RESHAPE`, `FLATTEN`, `COPY`, `MEAN`, `SUM`, `MIN`, `MAX`, `STD`, `APPEND`, `EXTEND`, `ITEMS`, `KEYS` and `VALUES` are standard array/container operations. `CAST` defaults to a new array; `FLATTEN` follows row-major order; axis arguments specify reductions. `TO_DEVICE`, `TO_CPU`, `TO_ARRAY`, `SCALAR_VALUE`, `ELEMENT_COUNT` and `TO_LIST` perform the named tensor/container conversions. `SET_MODE(model,TRAINING/EVALUATION)` switches neural-network behaviour.
- `LENGTH`, `SORT`, `ZIP`, `ENUMERATE`, `ABS`, `ALL`, `ANY`, `INTEGER`, `REAL`, `STRING`, `BOOLEAN`, `TUPLE`, `LIST`, `SET`, `MAP`, `GET_ATTRIBUTE`, `IS_INSTANCE` and `ROUND` are standard built-ins with the indicated roles. `super().__init__()` initializes the declared base class. `FORMAT` uses numbered placeholders and the explicitly retained numeric format specifications; `EMIT` writes a diagnostic. These are presentation operations, not hidden numerical algorithms.

### Standard-library primitive boundary

Calls under the following namespaces are explicitly delegated to standard libraries. Their named parameters and constants are shown at each call; all **lab-specific** procedures are defined in full below. Constructors, file objects, axes, video captures, optimizers and tensors retain the methods of their declared library. This fixes interpolation, boundary, dtype and random-state semantics without pretending to reimplement SVD, FFT, ORB, automatic differentiation or image codecs.

| Pseudocode namespace | Library / primitive family |
| --- | --- |
| `ARRAY` | NumPy arrays, broadcasting, linear algebra/SVD, FFT, statistics, RNG and window views |
| `CV` | OpenCV image conversion, chessboard detection/subpixel refinement, ORB/Hamming matching, RANSAC, morphology and watershed |
| `NDIMAGE`, `INTERPOLATE_COORDINATES` | SciPy ndimage filters, morphology and coordinate interpolation |
| `TENSOR`, `LAYERS` | PyTorch tensors, devices, differentiation, optimizers, layers and cross-entropy |
| `DATASET_INTERFACE`, `BATCH_LOADER` | PyTorch Dataset protocol and DataLoader batching, with the explicit local collation method |
| `READ_MNIST_IDX` | python-mnist IDX reader; `load_training` and `load_testing` return images and labels |
| `IMAGE` | Pillow image loading, mode conversion, resizing and encoding |
| `PATH` | Pathlib path joining, traversal, existence checks and output-directory creation; `/` joins path components for path operands |
| `PLOT` | Matplotlib figures, axes, drawing and saving; named arrays are the plotted data |
| `MATH`, `PROGRESS` | Standard scalar math and tqdm progress iteration |

`DATASET_INTERFACE` and `BATCH_LOADER` are standard abstractions; `MNISTDataset` and `MNISTClassifier`, where used, are fully defined below. Visualization calls preserve the figures' data, labels and output filenames. They can be translated to another renderer without changing the numerical procedure.

### Algorithm 1 — Setup — Environment and Configuration

```text
CALL ARRAY.set_printoptions(precision = 3, suppress = TRUE)
CALL EMIT("NumPy :", ARRAY.__version__)
CALL EMIT("OpenCV:", CV.__version__)
CALL EMIT("Setup : PASS")
```

### Algorithm 2 — 0.1 Locate the Lab Automatically

```text
PROCEDURE locate_lab_root()
    cwd ← PATH.cwd().resolve()
    FOR candidate IN [cwd, EXPAND cwd.parents] DO
        IF ((candidate / "data").is_dir() AND ((candidate / "notebooks") / "main.ipynb").is_file()) THEN
            RETURN candidate
        END IF
    END FOR
    RAISE FileNotFoundError("Could not locate the segmentation lab root.")
END PROCEDURE
LAB_ROOT ← locate_lab_root()
DATA_DIR ← (LAB_ROOT / "data")
OUTPUT_DIR ← ((LAB_ROOT / "outputs") / "figures")
CALL OUTPUT_DIR.mkdir(parents = TRUE, exist_ok = TRUE)
IMAGE_EXTENSIONS ← {".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp"}
DATA_FILES ← SORT(STREAM[path FOR path IN DATA_DIR.rglob("*") WHERE (path.is_file() AND (path.suffix.lower() IN IMAGE_EXTENSIONS))])
REQUIRE DATA_FILES OTHERWISE FORMAT("No supported images found in: {0}" , DATA_DIR)
PROCEDURE select_input_image(role)
    role ← role.lower()
    exact ← LIST[path FOR path IN DATA_FILES WHERE (path.stem.lower() = role)]
    IF exact THEN
        RETURN exact[0]
    END IF
    matches ← LIST[path FOR path IN DATA_FILES WHERE (role IN path.stem.lower())]
    IF (NOT matches) THEN
        RAISE FileNotFoundError(FORMAT("No image matching role '{0}' found in {1}" , role, DATA_DIR))
    END IF
    RETURN matches[0]
END PROCEDURE
INPUT_IMAGES ← MAP{role ↦ select_input_image(role) FOR role IN ("hand", "tower", "peppers")}
CALL EMIT("Lab root         :", LAB_ROOT)
CALL EMIT("Data dir         :", DATA_DIR)
CALL EMIT("Images discovered:", LENGTH(DATA_FILES))
CALL EMIT("Experiment roles :", LENGTH(INPUT_IMAGES))
CALL EMIT("Output           :", OUTPUT_DIR)
```

### Algorithm 3 — 0.2 Reusable helpers

```text
PROCEDURE load_grayscale_image(path)
    RETURN ARRAY.asarray(IMAGE.open(path).convert("L"), dtype = ARRAY.uint8)
END PROCEDURE
PROCEDURE load_rgb_image(path)
    RETURN ARRAY.asarray(IMAGE.open(path).convert("RGB"), dtype = ARRAY.uint8)
END PROCEDURE
PROCEDURE display_grayscale_image(ax, image, title)
    CALL ax.imshow(image, cmap = "gray")
    CALL ax.set_title(title)
    CALL ax.axis("off")
END PROCEDURE
PROCEDURE display_rgb_image(ax, image, title)
    CALL ax.imshow(image)
    CALL ax.set_title(title)
    CALL ax.axis("off")
END PROCEDURE
PROCEDURE save_output_figure(fig, filename)
    path ← (OUTPUT_DIR / filename)
    CALL fig.savefig(path, dpi = 160, bbox_inches = "tight")
    CALL EMIT("Saved:", path.name)
END PROCEDURE
PROCEDURE convert_mask_to_uint8(mask)
    RETURN CAST((CAST(ARRAY.asarray(mask), BOOLEAN) * 255), ARRAY.uint8)
END PROCEDURE
PROCEDURE overlay_segmentation_mask(image_rgb, mask, alpha = 0.35)
    output ← COPY(CAST(image_rgb, ARRAY.float32))
    mask_bool ← CAST(ARRAY.asarray(mask), BOOLEAN)
    red ← ARRAY.zeros_like(output)
    red[(…, 0)] ← 255
    output[mask_bool] ← (((1 − alpha) * output[mask_bool]) + (alpha * red[mask_bool]))
    RETURN CAST(ARRAY.clip(output, 0, 255), ARRAY.uint8)
END PROCEDURE
```

### Algorithm 4 — 1. Segmentation Problem Formulation

```text
PROCEDURE create_threshold_mask(image, threshold, foreground = "dark")
    image ← ARRAY.asarray(image)
    IF (image.ndim ≠ 2) THEN
        RAISE ValueError("create_threshold_mask expects a 2-D grayscale image.")
    END IF
    IF (foreground = "dark") THEN
        RETURN (image < threshold)
    END IF
    IF (foreground = "bright") THEN
        RETURN (image ≥ threshold)
    END IF
    RAISE ValueError("foreground must be 'dark' or 'bright'.")
END PROCEDURE
FOREGROUND ← TRUE
BACKGROUND ← FALSE
CALL EMIT("Mask convention:", {"foreground" ↦ FOREGROUND, "background" ↦ BACKGROUND})
```

### Algorithm 5 — 2. Load the Lab Images

```text
hand_image ← load_grayscale_image(INPUT_IMAGES["hand"])
tower_image_rgb ← load_rgb_image(INPUT_IMAGES["tower"])
peppers_image_rgb ← load_rgb_image(INPUT_IMAGES["peppers"])
(fig, axes) ← PLOT.subplots(1, 3, figsize = (15, 5))
CALL display_grayscale_image(axes[0], hand_image, "Hand")
CALL display_rgb_image(axes[1], tower_image_rgb, "Tower")
CALL display_rgb_image(axes[2], peppers_image_rgb, "Peppers")
CALL fig.tight_layout()
CALL save_output_figure(fig, "01_input_images.png")
CALL PLOT.show()
CALL EMIT("hand shape   :", hand_image.shape)
CALL EMIT("tower shape  :", tower_image_rgb.shape)
CALL EMIT("peppers shape:", peppers_image_rgb.shape)
```

### Algorithm 6 — 3. Histogram-Based Threshold Selection

```text
(fig, axes) ← PLOT.subplots(1, 2, figsize = (12, 4))
CALL display_grayscale_image(axes[0], hand_image, "Hand image")
CALL axes[1].hist(FLATTEN(hand_image), bins = 256, range = (0, 255))
CALL axes[1].set_title("Hand intensity histogram")
CALL axes[1].set_xlabel("Intensity")
CALL axes[1].set_ylabel("Pixel count")
CALL fig.tight_layout()
CALL save_output_figure(fig, "02_hand_histogram.png")
CALL PLOT.show()
```

### Algorithm 7 — 4. Manual Global Thresholding

```text
manual_threshold ← 120
manual_threshold_mask ← create_threshold_mask(hand_image, manual_threshold, foreground = "dark")
(fig, axes) ← PLOT.subplots(1, 3, figsize = (15, 5))
CALL display_grayscale_image(axes[0], hand_image, "Original")
CALL display_grayscale_image(axes[1], manual_threshold_mask, FORMAT("Mask — T={0}" , manual_threshold))
segmented_hand_image ← ARRAY.where(manual_threshold_mask, hand_image, 255)
CALL display_grayscale_image(axes[2], segmented_hand_image, "Segmented foreground")
CALL fig.tight_layout()
CALL save_output_figure(fig, "03_manual_threshold.png")
CALL PLOT.show()
```

### Algorithm 8 — 5. Threshold Sensitivity

```text
threshold_values ← [70, 100, 130, 160]
(fig, axes) ← PLOT.subplots(1, LENGTH(threshold_values), figsize = (16, 4))
FOR (ax, threshold) IN ZIP(axes, threshold_values) DO
    mask ← (hand_image < threshold)
    CALL display_grayscale_image(ax, mask, FORMAT("T={0}" , threshold))
END FOR
CALL fig.tight_layout()
CALL save_output_figure(fig, "04_threshold_sensitivity.png")
CALL PLOT.show()
```

### Algorithm 9 — 6. Otsu Thresholding

```text
(otsu_threshold, mask_otsu_cv) ← CV.threshold(hand_image, 0, 255, (CV.THRESH_BINARY_INV + CV.THRESH_OTSU))
otsu_mask ← (mask_otsu_cv > 0)
CALL EMIT("Otsu threshold:", otsu_threshold)
(fig, axes) ← PLOT.subplots(1, 3, figsize = (15, 5))
CALL display_grayscale_image(axes[0], hand_image, "Original")
CALL display_grayscale_image(axes[1], otsu_mask, FORMAT("Otsu mask — T={0:.1f}" , otsu_threshold))
segmentation_overlay ← overlay_segmentation_mask(ARRAY.stack(([hand_image] * 3), axis = (−1)), otsu_mask)
CALL display_rgb_image(axes[2], segmentation_overlay, "Mask overlay")
CALL fig.tight_layout()
CALL save_output_figure(fig, "05_otsu_threshold.png")
CALL PLOT.show()
```

### Algorithm 10 — 7. Gaussian Smoothing Before Thresholding

```text
smoothed_hand_image ← CV.GaussianBlur(hand_image, (5, 5), 0)
(blur_otsu_threshold, blur_otsu_cv) ← CV.threshold(smoothed_hand_image, 0, 255,
    (CV.THRESH_BINARY_INV + CV.THRESH_OTSU))
smoothed_otsu_mask ← (blur_otsu_cv > 0)
(fig, axes) ← PLOT.subplots(1, 3, figsize = (15, 5))
CALL display_grayscale_image(axes[0], hand_image, "Original")
CALL display_grayscale_image(axes[1], smoothed_hand_image, "Gaussian-smoothed")
CALL display_grayscale_image(axes[2], smoothed_otsu_mask, "Otsu after smoothing")
CALL fig.tight_layout()
CALL save_output_figure(fig, "06_smoothing_before_otsu.png")
CALL PLOT.show()
```

### Algorithm 11 — 8. Adaptive Thresholding

```text
adaptive_mean_mask ← CV.adaptiveThreshold(hand_image, 255, CV.ADAPTIVE_THRESH_MEAN_C, CV.THRESH_BINARY_INV,
    31, 5)
adaptive_gaussian_mask ← CV.adaptiveThreshold(hand_image, 255, CV.ADAPTIVE_THRESH_GAUSSIAN_C,
    CV.THRESH_BINARY_INV, 31, 5)
(fig, axes) ← PLOT.subplots(1, 3, figsize = (15, 5))
CALL display_grayscale_image(axes[0], hand_image, "Original")
CALL display_grayscale_image(axes[1], adaptive_mean_mask, "Adaptive mean")
CALL display_grayscale_image(axes[2], adaptive_gaussian_mask, "Adaptive Gaussian")
CALL fig.tight_layout()
CALL save_output_figure(fig, "07_adaptive_thresholding.png")
CALL PLOT.show()
```

### Algorithm 12 — 9. Morphological Processing

```text
PROCEDURE apply_morphological_operation(mask, operation, kernel, iterations = 1)
    mask_u8 ← convert_mask_to_uint8(mask)
    IF (iterations < 1) THEN
        RAISE ValueError("iterations must be at least 1.")
    END IF
    IF (operation = "erode") THEN
        result ← CV.erode(mask_u8, kernel, iterations = iterations)
    ELSE
        IF (operation = "dilate") THEN
            result ← CV.dilate(mask_u8, kernel, iterations = iterations)
        ELSE
            IF (operation = "open") THEN
                result ← CV.morphologyEx(mask_u8, CV.MORPH_OPEN, kernel, iterations = iterations)
            ELSE
                IF (operation = "close") THEN
                    result ← CV.morphologyEx(mask_u8, CV.MORPH_CLOSE, kernel, iterations = iterations)
                ELSE
                    RAISE ValueError("operation must be 'erode', 'dilate', 'open', or 'close'.")
                END IF
            END IF
        END IF
    END IF
    RETURN (result > 0)
END PROCEDURE
CALL EMIT("Supported morphology:", ["erode", "dilate", "open", "close"])
```

### Algorithm 13 — 10. Structuring Elements

```text
kernel_rect ← CV.getStructuringElement(CV.MORPH_RECT, (7, 7))
kernel_ellipse ← CV.getStructuringElement(CV.MORPH_ELLIPSE, (7, 7))
kernel_cross ← CV.getStructuringElement(CV.MORPH_CROSS, (7, 7))
(fig, axes) ← PLOT.subplots(1, 3, figsize = (10, 3))
CALL display_grayscale_image(axes[0], kernel_rect, "Rectangle")
CALL display_grayscale_image(axes[1], kernel_ellipse, "Ellipse")
CALL display_grayscale_image(axes[2], kernel_cross, "Cross")
CALL fig.tight_layout()
CALL save_output_figure(fig, "08_structuring_elements.png")
CALL PLOT.show()
```

### Algorithm 14 — 11. Erosion and Dilation

```text
binary_mask_uint8 ← convert_mask_to_uint8(smoothed_otsu_mask)
kernel ← CV.getStructuringElement(CV.MORPH_ELLIPSE, (7, 7))
eroded_mask ← CV.erode(binary_mask_uint8, kernel, iterations = 1)
dilated_mask ← CV.dilate(binary_mask_uint8, kernel, iterations = 1)
(fig, axes) ← PLOT.subplots(1, 3, figsize = (15, 5))
CALL display_grayscale_image(axes[0], binary_mask_uint8, "Original mask")
CALL display_grayscale_image(axes[1], eroded_mask, "Erosion")
CALL display_grayscale_image(axes[2], dilated_mask, "Dilation")
CALL fig.tight_layout()
CALL save_output_figure(fig, "09_erosion_dilation.png")
CALL PLOT.show()
```

### Algorithm 15 — 12. Opening and Closing

```text
opened_mask ← CV.morphologyEx(binary_mask_uint8, CV.MORPH_OPEN, kernel)
closed_mask ← CV.morphologyEx(binary_mask_uint8, CV.MORPH_CLOSE, kernel)
(fig, axes) ← PLOT.subplots(1, 3, figsize = (15, 5))
CALL display_grayscale_image(axes[0], binary_mask_uint8, "Original mask")
CALL display_grayscale_image(axes[1], opened_mask, "Opening")
CALL display_grayscale_image(axes[2], closed_mask, "Closing")
CALL fig.tight_layout()
CALL save_output_figure(fig, "10_opening_closing.png")
CALL PLOT.show()
```

### Algorithm 16 — 13. Morphological Gradient

```text
morphological_gradient ← CV.morphologyEx(binary_mask_uint8, CV.MORPH_GRADIENT, kernel)
(fig, axes) ← PLOT.subplots(1, 2, figsize = (10, 5))
CALL display_grayscale_image(axes[0], binary_mask_uint8, "Mask")
CALL display_grayscale_image(axes[1], morphological_gradient, "Morphological gradient")
CALL fig.tight_layout()
CALL save_output_figure(fig, "11_morphological_gradient.png")
CALL PLOT.show()
```

### Algorithm 17 — 14. Hole Filling

```text
binary_mask_with_holes ← CAST(smoothed_otsu_mask, BOOLEAN)
hole_filled_mask ← NDIMAGE.binary_fill_holes(binary_mask_with_holes)
(fig, axes) ← PLOT.subplots(1, 3, figsize = (15, 5))
CALL display_grayscale_image(axes[0], binary_mask_with_holes, "Before filling")
CALL display_grayscale_image(axes[1], hole_filled_mask, "After filling")
CALL display_grayscale_image(axes[2], (CAST(hole_filled_mask, INTEGER) − CAST(binary_mask_with_holes,
    INTEGER)), "Pixels added")
CALL fig.tight_layout()
CALL save_output_figure(fig, "12_hole_filling.png")
CALL PLOT.show()
```

### Algorithm 18 — 15. Connected Components

```text
(num_labels, labels, stats, centroids) ← CV.connectedComponentsWithStats(convert_mask_to_uint8(hole_filled_mask),
    connectivity = 8)
CALL EMIT("Number of foreground components:", (num_labels − 1))
component_areas ← stats[(1:, CV.CC_STAT_AREA)]
CALL EMIT("Foreground areas:", component_areas)
(fig, axes) ← PLOT.subplots(1, 2, figsize = (12, 5))
CALL display_grayscale_image(axes[0], hole_filled_mask, "Binary mask")
CALL axes[1].imshow(labels, cmap = "nipy_spectral")
CALL axes[1].set_title("Connected-component labels")
CALL axes[1].axis("off")
CALL fig.tight_layout()
CALL save_output_figure(fig, "13_connected_components.png")
CALL PLOT.show()
```

### Algorithm 19 — 16. Remove Small Components

```text
minimum_area ← 500
area_filtered_components ← ARRAY.zeros_like(labels, dtype = BOOLEAN)
FOR label_id IN RANGE(1, num_labels) DO
    area ← stats[(label_id, CV.CC_STAT_AREA)]
    IF (area ≥ minimum_area) THEN
        area_filtered_components ← (area_filtered_components OR_EACH (labels = label_id))
    END IF
END FOR
(fig, axes) ← PLOT.subplots(1, 2, figsize = (10, 5))
CALL display_grayscale_image(axes[0], hole_filled_mask, "Before area filtering")
CALL display_grayscale_image(axes[1], area_filtered_components, FORMAT("Area ≥ {0}" , minimum_area))
CALL fig.tight_layout()
CALL save_output_figure(fig, "14_component_area_filtering.png")
CALL PLOT.show()
```

### Algorithm 20 — 17. Contours

```text
(contours, hierarchy) ← CV.findContours(convert_mask_to_uint8(area_filtered_components), CV.RETR_EXTERNAL,
    CV.CHAIN_APPROX_SIMPLE)
hand_rgb ← ARRAY.stack(([hand_image] * 3), axis = (−1))
contour_overlay ← COPY(hand_rgb)
CALL CV.drawContours(contour_overlay, contours, (−1), (255, 0, 0), 2)
(fig, axes) ← PLOT.subplots(1, 2, figsize = (10, 5))
CALL display_grayscale_image(axes[0], area_filtered_components, "Clean mask")
CALL display_rgb_image(axes[1], contour_overlay, "Detected contours")
CALL fig.tight_layout()
CALL save_output_figure(fig, "15_contours.png")
CALL PLOT.show()
CALL EMIT("Number of external contours:", LENGTH(contours))
```

### Algorithm 21 — 18. Region Properties

```text
region_measurements ← []
FOR (index, contour) IN ENUMERATE(contours, start = 1) DO
    area ← CV.contourArea(contour)
    perimeter ← CV.arcLength(contour, TRUE)
    (x, y, w, h) ← CV.boundingRect(contour)
    moments ← CV.moments(contour)
    IF (moments["m00"] ≠ 0) THEN
        cx ← (moments["m10"] / moments["m00"])
        cy ← (moments["m01"] / moments["m00"])
    ELSE
        cx ← ARRAY.nan
        cy ← ARRAY.nan
    END IF
    circularity ← (IF (perimeter > 0) THEN (((4 * ARRAY.pi) * area) / (perimeter ^ 2)) ELSE ARRAY.nan)
    aspect_ratio ← (IF (h > 0) THEN (w / h) ELSE ARRAY.nan)
    CALL APPEND(region_measurements, {"component" ↦ index, "area" ↦ area, "perimeter" ↦ perimeter, "cx" ↦ cx,
        "cy" ↦ cy, "width" ↦ w, "height" ↦ h, "aspect_ratio" ↦ aspect_ratio, "circularity" ↦ circularity})
END FOR
FOR row IN region_measurements DO
    CALL EMIT(row)
END FOR
```

### Algorithm 22 — 19. Color Segmentation

```text
peppers_image_hsv ← CV.cvtColor(peppers_image_rgb, CV.COLOR_RGB2HSV)
hue ← peppers_image_hsv[(…, 0)]
saturation ← peppers_image_hsv[(…, 1)]
value ← peppers_image_hsv[(…, 2)]
(fig, axes) ← PLOT.subplots(1, 4, figsize = (16, 4))
CALL display_rgb_image(axes[0], peppers_image_rgb, "RGB")
CALL display_grayscale_image(axes[1], hue, "Hue")
CALL display_grayscale_image(axes[2], saturation, "Saturation")
CALL display_grayscale_image(axes[3], value, "Value")
CALL fig.tight_layout()
CALL save_output_figure(fig, "16_hsv_channels.png")
CALL PLOT.show()
```

### Algorithm 23 — HSV Range Experiment

```text
lower_red_1 ← ARRAY.array([0, 80, 50], dtype = ARRAY.uint8)
upper_red_1 ← ARRAY.array([12, 255, 255], dtype = ARRAY.uint8)
lower_red_2 ← ARRAY.array([165, 80, 50], dtype = ARRAY.uint8)
upper_red_2 ← ARRAY.array([179, 255, 255], dtype = ARRAY.uint8)
mask_red_1 ← CV.inRange(peppers_image_hsv, lower_red_1, upper_red_1)
mask_red_2 ← CV.inRange(peppers_image_hsv, lower_red_2, upper_red_2)
red_region_mask ← ((mask_red_1 > 0) OR_EACH (mask_red_2 > 0))
segmented_red_regions ← COPY(peppers_image_rgb)
segmented_red_regions[(COMPLEMENT red_region_mask)] ← 0
(fig, axes) ← PLOT.subplots(1, 3, figsize = (15, 5))
CALL display_rgb_image(axes[0], peppers_image_rgb, "Original peppers")
CALL display_grayscale_image(axes[1], red_region_mask, "Red-color mask")
CALL display_rgb_image(axes[2], segmented_red_regions, "Segmented red regions")
CALL fig.tight_layout()
CALL save_output_figure(fig, "17_color_segmentation.png")
CALL PLOT.show()
```

### Algorithm 24 — 20. Edge-Based Segmentation

```text
tower_image_gray ← CV.cvtColor(tower_image_rgb, CV.COLOR_RGB2GRAY)
smoothed_tower_image ← CV.GaussianBlur(tower_image_gray, (5, 5), 0)
tower_edge_map ← CV.Canny(smoothed_tower_image, 80, 160)
edge_kernel ← CV.getStructuringElement(CV.MORPH_RECT, (5, 5))
closed_tower_edge_map ← CV.morphologyEx(tower_edge_map, CV.MORPH_CLOSE, edge_kernel, iterations = 2)
(fig, axes) ← PLOT.subplots(1, 3, figsize = (15, 5))
CALL display_rgb_image(axes[0], tower_image_rgb, "Tower")
CALL display_grayscale_image(axes[1], tower_edge_map, "Canny edges")
CALL display_grayscale_image(axes[2], closed_tower_edge_map, "Closed edge map")
CALL fig.tight_layout()
CALL save_output_figure(fig, "18_edge_based_segmentation.png")
CALL PLOT.show()
```

### Algorithm 25 — 21. Distance Transform

```text
distance_transform_map ← CV.distanceTransform(convert_mask_to_uint8(area_filtered_components), CV.DIST_L2,
    5)
(fig, axes) ← PLOT.subplots(1, 2, figsize = (10, 5))
CALL display_grayscale_image(axes[0], area_filtered_components, "Binary mask")
im ← axes[1].imshow(distance_transform_map, cmap = "viridis")
CALL axes[1].set_title("Distance transform")
CALL axes[1].axis("off")
CALL fig.colorbar(im, ax = axes[1], fraction = 0.046)
CALL fig.tight_layout()
CALL save_output_figure(fig, "19_distance_transform.png")
CALL PLOT.show()
```

### Algorithm 26 — 22. Watershed Segmentation

```text
watershed_source_image ← COPY(peppers_image_rgb)
peppers_image_gray ← CV.cvtColor(peppers_image_rgb, CV.COLOR_RGB2GRAY)
(_, peppers_binary) ← CV.threshold(peppers_image_gray, 0, 255, (CV.THRESH_BINARY + CV.THRESH_OTSU))
ws_kernel ← ARRAY.ones((3, 3), ARRAY.uint8)
opening ← CV.morphologyEx(peppers_binary, CV.MORPH_OPEN, ws_kernel, iterations = 2)
sure_background ← CV.dilate(opening, ws_kernel, iterations = 3)
distance_transform ← CV.distanceTransform(opening, CV.DIST_L2, 5)
(_, sure_foreground) ← CV.threshold(distance_transform, (0.5 * MAX(distance_transform)), 255, 0)
sure_foreground ← ARRAY.uint8(sure_foreground)
unknown_region ← CV.subtract(sure_background, sure_foreground)
(num_markers, markers) ← CV.connectedComponents(sure_foreground)
markers ← (markers + 1)
markers[(unknown_region = 255)] ← 0
watershed_result_labels ← CV.watershed(CV.cvtColor(watershed_source_image, CV.COLOR_RGB2BGR), COPY(markers))
watershed_boundary_overlay ← COPY(watershed_source_image)
watershed_boundary_overlay[(watershed_result_labels = (−1))] ← [255, 0, 0]
(fig, axes) ← PLOT.subplots(2, 3, figsize = (15, 10))
CALL display_rgb_image(axes[(0, 0)], peppers_image_rgb, "Input")
CALL display_grayscale_image(axes[(0, 1)], opening, "Opening")
CALL display_grayscale_image(axes[(0, 2)], sure_background, "Sure background")
CALL display_grayscale_image(axes[(1, 0)], distance_transform, "Distance transform")
CALL display_grayscale_image(axes[(1, 1)], sure_foreground, "Sure foreground")
CALL display_rgb_image(axes[(1, 2)], watershed_boundary_overlay, "Watershed boundaries")
CALL fig.tight_layout()
CALL save_output_figure(fig, "20_watershed.png")
CALL PLOT.show()
```

### Algorithm 27 — 23. Ground Truth and Segmentation Metrics

```text
PROCEDURE compute_segmentation_metrics(ground_truth, prediction)
    ground_truth_mask ← CAST(ARRAY.asarray(ground_truth), BOOLEAN)
    predicted_mask ← CAST(ARRAY.asarray(prediction), BOOLEAN)
    true_positives ← SUM(ARRAY.logical_and(ground_truth_mask, predicted_mask))
    true_negatives ← SUM(ARRAY.logical_and((COMPLEMENT ground_truth_mask), (COMPLEMENT predicted_mask)))
    false_positives ← SUM(ARRAY.logical_and((COMPLEMENT ground_truth_mask), predicted_mask))
    false_negatives ← SUM(ARRAY.logical_and(ground_truth_mask, (COMPLEMENT predicted_mask)))
    epsilon ← 1e-12
    accuracy ← ((true_positives + true_negatives) / ((((true_positives + true_negatives) + false_positives) + false_negatives) + epsilon))
    precision ← (true_positives / ((true_positives + false_positives) + epsilon))
    recall ← (true_positives / ((true_positives + false_negatives) + epsilon))
    iou ← (true_positives / (((true_positives + false_positives) + false_negatives) + epsilon))
    dice ← ((2 * true_positives) / ((((2 * true_positives) + false_positives) + false_negatives) + epsilon))
    RETURN {"TP" ↦ INTEGER(true_positives), "TN" ↦ INTEGER(true_negatives), "FP" ↦ INTEGER(false_positives),
        "FN" ↦ INTEGER(false_negatives), "Accuracy" ↦ REAL(accuracy), "Precision" ↦ REAL(precision),
        "Recall" ↦ REAL(recall), "IoU" ↦ REAL(iou), "Dice" ↦ REAL(dice)}
END PROCEDURE
```

### Algorithm 28 — Synthetic Ground-Truth Check

```text
demo_gt ← COPY(area_filtered_components)
demo_prediction ← (CV.erode(convert_mask_to_uint8(area_filtered_components), ARRAY.ones((7, 7), ARRAY.uint8),
    iterations = 1) > 0)
demo_metrics ← compute_segmentation_metrics(demo_gt, demo_prediction)
FOR (key, value) IN ITEMS(demo_metrics) DO
    IF IS_INSTANCE(value, REAL) THEN
        CALL EMIT(FORMAT("{0:10s}: {1:.4f}" , key, value))
    ELSE
        CALL EMIT(FORMAT("{0:10s}: {1}" , key, value))
    END IF
END FOR
```

### Algorithm 29 — 24. Dice vs IoU Relationship

```text
demo_iou ← demo_metrics["IoU"]
demo_dice ← demo_metrics["Dice"]
dice_from_iou ← ((2.0 * demo_iou) / (1.0 + demo_iou))
iou_from_dice ← (demo_dice / (2.0 − demo_dice))
REQUIRE ARRAY.isclose(dice_from_iou, demo_dice) OTHERWISE STOP WITH ValidationError
REQUIRE ARRAY.isclose(iou_from_dice, demo_iou) OTHERWISE STOP WITH ValidationError
CALL EMIT(FORMAT("IoU           : {0:.4f}" , demo_iou))
CALL EMIT(FORMAT("Dice          : {0:.4f}" , demo_dice))
CALL EMIT(FORMAT("Dice from IoU : {0:.4f}" , dice_from_iou))
CALL EMIT(FORMAT("IoU from Dice : {0:.4f}" , iou_from_dice))
```

### Algorithm 30 — 25. Under-Segmentation vs Over-Segmentation

```text
comparison_kernel ← ARRAY.ones((9, 9), dtype = ARRAY.uint8)
under_segmented ← (CV.erode(convert_mask_to_uint8(demo_gt), comparison_kernel, iterations = 1) > 0)
over_segmented ← (CV.dilate(convert_mask_to_uint8(demo_gt), comparison_kernel, iterations = 1) > 0)
under_metrics ← compute_segmentation_metrics(demo_gt, under_segmented)
over_metrics ← compute_segmentation_metrics(demo_gt, over_segmented)
CALL EMIT("Under-segmentation:", FORMAT("Precision={0:.4f}" , under_metrics["Precision"]),
    FORMAT("Recall={0:.4f}" , under_metrics["Recall"]), FORMAT("IoU={0:.4f}" , under_metrics["IoU"]))
CALL EMIT("Over-segmentation:", FORMAT("Precision={0:.4f}" , over_metrics["Precision"]),
    FORMAT("Recall={0:.4f}" , over_metrics["Recall"]), FORMAT("IoU={0:.4f}" , over_metrics["IoU"]))
```

### Algorithm 31 — 26. End-to-End Binary Segmentation Pipeline

```text
PROCEDURE segment_dark_foreground_object(image_gray, blur_kernel = (5, 5), minimum_area = 500)
    blurred ← CV.GaussianBlur(image_gray, blur_kernel, 0)
    (threshold_value, binary) ← CV.threshold(blurred, 0, 255, (CV.THRESH_BINARY_INV + CV.THRESH_OTSU))
    kernel ← CV.getStructuringElement(CV.MORPH_ELLIPSE, (5, 5))
    cleaned ← apply_morphological_operation((binary > 0), operation = "open", kernel = kernel,
        iterations = 1)
    cleaned ← apply_morphological_operation(cleaned, operation = "close", kernel = kernel, iterations = 1)
    hole_filled_mask ← NDIMAGE.binary_fill_holes(cleaned)
    (num_labels, labels, stats, _) ← CV.connectedComponentsWithStats(convert_mask_to_uint8(hole_filled_mask),
        connectivity = 8)
    final_mask ← ARRAY.zeros_like(hole_filled_mask, dtype = BOOLEAN)
    FOR label_id IN RANGE(1, num_labels) DO
        area ← stats[(label_id, CV.CC_STAT_AREA)]
        IF (area ≥ minimum_area) THEN
            final_mask ← (final_mask OR_EACH (labels = label_id))
        END IF
    END FOR
    RETURN {"threshold" ↦ REAL(threshold_value), "blurred" ↦ blurred, "binary" ↦ (binary > 0),
        "cleaned" ↦ cleaned, "filled" ↦ hole_filled_mask, "mask" ↦ final_mask}
END PROCEDURE
```

### Algorithm 32 — 27. Segmentation Method Selection Criteria

```text
PROCEDURE recommend_segmentation_strategy(uneven_illumination = FALSE, color_is_discriminative = FALSE,
    touching_objects = FALSE)
    IF touching_objects THEN
        RETURN "distance transform + marker-controlled watershed"
    END IF
    IF color_is_discriminative THEN
        RETURN "color-space thresholding (for example HSV)"
    END IF
    IF uneven_illumination THEN
        RETURN "adaptive thresholding"
    END IF
    RETURN "global thresholding / Otsu"
END PROCEDURE
selection_examples ← {"uniform grayscale object" ↦ recommend_segmentation_strategy(),
    "uneven lighting" ↦ recommend_segmentation_strategy(uneven_illumination = TRUE),
    "distinctive color" ↦ recommend_segmentation_strategy(color_is_discriminative = TRUE),
    "touching objects" ↦ recommend_segmentation_strategy(touching_objects = TRUE)}
FOR (case, method) IN ITEMS(selection_examples) DO
    CALL EMIT(FORMAT("{0:24s} -> {1}" , case, method))
END FOR
```

### Algorithm 33 — 28. Integrated Segmentation Workflow

```text
hand_result ← segment_dark_foreground_object(hand_image, minimum_area = 500)
final_hand_mask ← hand_result["mask"]
final_overlay ← overlay_segmentation_mask(ARRAY.stack(([hand_image] * 3), axis = (−1)), final_hand_mask)
(fig, axes) ← PLOT.subplots(2, 3, figsize = (15, 10))
CALL display_grayscale_image(axes[(0, 0)], hand_image, "Input")
CALL display_grayscale_image(axes[(0, 1)], hand_result["blurred"], "Smoothed")
CALL display_grayscale_image(axes[(0, 2)], hand_result["binary"], "Otsu threshold")
CALL display_grayscale_image(axes[(1, 0)], hand_result["cleaned"], "Morphological cleanup")
CALL display_grayscale_image(axes[(1, 1)], final_hand_mask, "Final mask")
CALL display_rgb_image(axes[(1, 2)], final_overlay, "Final overlay")
CALL fig.tight_layout()
CALL save_output_figure(fig, "21_complete_pipeline.png")
CALL PLOT.show()
CALL EMIT("Pipeline Otsu threshold:", hand_result["threshold"])
REQUIRED_OUTPUTS ← ["01_input_images.png", "02_hand_histogram.png", "03_manual_threshold.png", "04_threshold_sensitivity.png",
    "05_otsu_threshold.png", "06_smoothing_before_otsu.png", "07_adaptive_thresholding.png", "08_structuring_elements.png",
    "09_erosion_dilation.png", "10_opening_closing.png", "11_morphological_gradient.png", "12_hole_filling.png",
    "13_connected_components.png", "14_component_area_filtering.png", "15_contours.png", "16_hsv_channels.png",
    "17_color_segmentation.png", "18_edge_based_segmentation.png", "19_distance_transform.png", "20_watershed.png",
    "21_complete_pipeline.png"]
missing_outputs ← LIST[output_name FOR output_name IN REQUIRED_OUTPUTS WHERE (NOT (OUTPUT_DIR / output_name).exists())]
IF missing_outputs THEN
    RAISE FileNotFoundError(("Missing outputs: " + ", ".join(missing_outputs)))
END IF
CALL EMIT(FORMAT("Output validation passed: {0} files." , LENGTH(REQUIRED_OUTPUTS)))
```

### Completeness check

| Executable element | Included in the pseudocode |
| --- | ---: |
| Source code cells | 33 |
| Procedure bodies | 14 |
| Assignments | 169 |
| Conditional branches | 19 |
| Loops | 8 |
| Assertions | 3 |
| Explicit error branches | 7 |

The counts check transcription coverage, not mathematical correctness or a successful execution. Every original diagnostic and rendering call is included. The limitations discussed below still apply.

## Task Coverage

Every task defined in the Problem Statement is covered by the coding sequence above:

- **Task 1:** Segmentation Problem Formulation
- **Task 2:** Load the Lab Images
- **Task 3:** Histogram-Based Threshold Selection
- **Task 4:** Manual Global Thresholding
- **Task 5:** Threshold Sensitivity
- **Task 6:** Otsu Thresholding
- **Task 7:** Gaussian Smoothing Before Thresholding
- **Task 8:** Adaptive Thresholding
- **Task 9:** Morphological Processing
- **Task 10:** Structuring Elements
- **Task 11:** Erosion and Dilation
- **Task 12:** Opening and Closing
- **Task 13:** Morphological Gradient
- **Task 14:** Hole Filling
- **Task 15:** Connected Components
- **Task 16:** Remove Small Components
- **Task 17:** Contours
- **Task 18:** Region Properties
- **Task 19:** Color Segmentation
- **Task 20:** Edge-Based Segmentation
- **Task 21:** Distance Transform
- **Task 22:** Watershed Segmentation
- **Task 23:** Ground Truth and Segmentation Metrics
- **Task 24:** Dice vs IoU Relationship
- **Task 25:** Under-Segmentation vs Over-Segmentation
- **Task 26:** End-to-End Binary Segmentation Pipeline
- **Task 27:** Segmentation Method Selection Criteria
- **Task 28:** Integrated Segmentation Workflow


## Notebook Relationship

- `problem_statement.ipynb` defines **what must be solved**.
- `theory.ipynb` explains **why the methods work**.
- `requirements.ipynb` defines **what is needed to run the lab**.
- `algorithm.ipynb` defines **how to build the solution step by step**.
- `main.ipynb` is the **executable implementation**.


## Required Output Contract

The current `REQUIRED_OUTPUTS` list in `main.ipynb` contains 21 files under `../outputs/figures/`:

- `01_input_images.png`
- `02_hand_histogram.png`
- `03_manual_threshold.png`
- `04_threshold_sensitivity.png`
- `05_otsu_threshold.png`
- `06_smoothing_before_otsu.png`
- `07_adaptive_thresholding.png`
- `08_structuring_elements.png`
- `09_erosion_dilation.png`
- `10_opening_closing.png`
- `11_morphological_gradient.png`
- `12_hole_filling.png`
- `13_connected_components.png`
- `14_component_area_filtering.png`
- `15_contours.png`
- `16_hsv_channels.png`
- `17_color_segmentation.png`
- `18_edge_based_segmentation.png`
- `19_distance_transform.png`
- `20_watershed.png`
- `21_complete_pipeline.png`

A presence check detects missing artifacts, not stale artifacts. Inspect the figures from the run being reported.

## Implementation Traceability

Reviewed against [source revision 66078c2](https://github.com/denoskume/msc-coro-dassip-portfolio/tree/66078c2481115d0b2596c6926e0750d136ba1005/labs/image-processing/segmentation/notebooks).

The following helpers/classes already exist in `main.ipynb`; algorithm operations that are not listed here are inline notebook cells or descriptive pseudocode:

- `locate_lab_root`
- `select_input_image`
- `load_grayscale_image`
- `load_rgb_image`
- `display_grayscale_image`
- `display_rgb_image`
- `save_output_figure`
- `convert_mask_to_uint8`
- `overlay_segmentation_mask`
- `create_threshold_mask`
- `apply_morphological_operation`
- `compute_segmentation_metrics`
- `segment_dark_foreground_object`
- `recommend_segmentation_strategy`

Examples of **existing runtime assertions**, copied from the implementation:

```python
assert DATA_FILES
assert np.isclose(dice_from_iou, demo_dice)
assert np.isclose(iou_from_dice, demo_iou)
```

This excerpt is not a new test suite and does not list every runtime exception.

## Implementation Limits and Interpretation

- Mask-derived reference examples demonstrate metric behavior; they are not independently annotated ground truth.
- The current metric helper relies on callers for matching shapes and uses an epsilon convention for empty denominators. State that convention when interpreting empty masks.
- The recommendation helper returns text according to scene-property flags; it does not dispatch the selected segmentation method.
- Region and edge representations differ: contour area, component pixel count, boundary maps and watershed labels are not interchangeable metrics.

## 33. End

The coding plan ends after the implemented checks and required artifacts have been reviewed. Interpret outputs beside their corresponding experiment, then write the overall analysis at the bottom of `main.ipynb`. Report the implementation limits below; a numerical pass alone is not proof of physical validity or application-level accuracy.